### Validate Google Maps POIs with OSM

In [ ]:
import osmnx as ox
import pandas as pd
import geopandas as gpd
import os

from tqdm import tqdm
tqdm.pandas

In [10]:
pilot_poly = gpd.read_file("../data/pilot_lad.geojson")
pilot_poly

,FID,LAD23CD,LAD23NM,LAD23NMW,BNG_E,BNG_N,LONG,LAT,GlobalID,area,geom_split,geometry
0,17,E06000017,Rutland,,492992,308655,-0.62630,52.66765,4403b9ad-da3a-4240-9c9e-a396071bc4b0,0.052322,"POLYGON ((-0.609443616633852 52.7597305919592,...","POLYGON ((-0.60944 52.75973, -0.60909 52.7597,..."
1,50,E06000053,Isles of Scilly,,91327,11447,-6.30217,49.92332,afca461c-801c-4df1-b0f1-167b46294e75,0.002857,"POLYGON ((-6.39884611349852 49.8656908692221, ...","MULTIPOLYGON (((-6.39885 49.86569, -6.39874 49..."
2,50,E06000053,Isles of Scilly,,91327,11447,-6.30217,49.92332,afca461c-801c-4df1-b0f1-167b46294e75,0.002857,"POLYGON ((-6.39730197263228 49.8664839207227, ...","MULTIPOLYGON (((-6.39885 49.86569, -6.39874 49..."
3,50,E06000053,Isles of Scilly,,91327,11447,-6.30217,49.92332,afca461c-801c-4df1-b0f1-167b46294e75,0.002857,"POLYGON ((-6.38574208860896 49.8667928355807, ...","MULTIPOLYGON (((-6.39885 49.86569, -6.39874 49..."
4,50,E06000053,Isles of Scilly,,91327,11447,-6.30217,49.92332,afca461c-801c-4df1-b0f1-167b46294e75,0.002857,"POLYGON ((-6.39287242297511 49.8685893414364, ...","MULTIPOLYGON (((-6.39885 49.86569, -6.39874 49..."
...,...,...,...,...,...,...,...,...,...,...,...,...
65,251,E08000025,Birmingham,,408150,287352,-1.88141,52.48404,06b4cfe9-de02-434d-9afa-4186e208111e,0.035453,"POLYGON ((-1.82482002022059 52.6077840088598, ...","POLYGON ((-1.82482 52.60778, -1.82374 52.60747..."
66,261,E08000035,Leeds,,432528,436384,-1.50736,53.82273,25f8a824-ef47-4eda-aabe-1de93fb85053,0.075328,"POLYGON ((-1.340565695099 53.9448842475508, -1...","POLYGON ((-1.34057 53.94488, -1.34074 53.94468..."
67,264,E09000001,City of London,,532382,181358,-0.09351,51.51564,3423eef3-091e-4798-aad0-338669589b08,0.000408,POLYGON ((-0.0966877822838667 51.5231942024718...,"POLYGON ((-0.09669 51.52319, -0.09668 51.52317..."
68,282,E09000019,Islington,,531160,184645,-0.10989,51.54546,52b636ec-aec7-48b3-a3ae-648395ee11a4,0.001926,"POLYGON ((-0.119269904958699 51.5750935045979,...","POLYGON ((-0.11927 51.57509, -0.1192 51.57499,..."


In [15]:
# get all museum building in Greater London
# `True` means retrieve any object with this tag, regardless of value

tags = {"tourism": "museum"}
osm_pois = []

for _,r in pilot_poly.iterrows():
    gdf = ox.features_from_polygon(pilot_poly.geometry.iloc[_], tags)
    osm_pois.append(gdf.assign(LAD23NM = r.LAD23NM))

/opt/miniconda3/envs/thesis/lib/python3.11/site-packages/shapely/predicates.py:878: RuntimeWarning: invalid value encountered in intersects
  return lib.intersects(a, b, **kwargs)
/opt/miniconda3/envs/thesis/lib/python3.11/site-packages/shapely/set_operations.py:451: RuntimeWarning: invalid value encountered in union
  return lib.union(a, b, **kwargs)
/opt/miniconda3/envs/thesis/lib/python3.11/site-packages/shapely/predicates.py:878: RuntimeWarning: invalid value encountered in intersects
  return lib.intersects(a, b, **kwargs)
/opt/miniconda3/envs/thesis/lib/python3.11/site-packages/shapely/set_operations.py:451: RuntimeWarning: invalid value encountered in union
  return lib.union(a, b, **kwargs)
/opt/miniconda3/envs/thesis/lib/python3.11/site-packages/shapely/predicates.py:878: RuntimeWarning: invalid value encountered in intersects
  return lib.intersects(a, b, **kwargs)
/opt/miniconda3/envs/thesis/lib/python3.11/site-packages/shapely/set_operations.py:451: RuntimeWarning: invalid 

In [33]:
osm_pois_df = pd.concat(osm_pois).reset_index().drop_duplicates("id")
osm_pois_df.LAD23NM.value_counts()

LAD23NM
North Yorkshire            45
Northumberland             41
Westmorland and Furness    27
Birmingham                 19
Leeds                      17
Kensington and Chelsea     12
Islington                   9
City of London              6
Rutland                     3
Isles of Scilly             3
Name: count, dtype: int64

In [34]:
osm_pois_df.building.value_counts()

building
yes              79
heritage          5
public            3
industrial        2
civic             2
museum            1
retail            1
barn              1
hut               1
warehouse         1
house             1
university        1
country_house     1
Name: count, dtype: int64

In [35]:
osm_pois_df.element.value_counts()

element
way         111
node         70
relation      1
Name: count, dtype: int64

In [50]:
osm_pois_df.loc[osm_pois_df.element=='node', 'type'].info()

<class 'pandas.core.series.Series'>
Index: 70 entries, 3 to 656
Series name: type
Non-Null Count  Dtype 
--------------  ----- 
0 non-null      object
dtypes: object(1)
memory usage: 1.1+ KB


In [39]:
list(osm_pois_df.columns)

['element',
 'id',
 'geometry',
 'building',
 'name',
 'tourism',
 'wikidata',
 'addr:city',
 'addr:country',
 'addr:postcode',
 'addr:street',
 'alt_name',
 'email',
 'source',
 'website',
 'wikipedia',
 'LAD23NM',
 'booth',
 'covered',
 'disused:amenity',
 'man_made',
 'operator',
 'fee',
 'museum',
 'addr:county',
 'addr:suburb',
 'addr:village',
 'level',
 'addr:housename',
 'survey:date',
 'addr:housenumber',
 'opening_hours',
 'phone',
 'fixme',
 'check_date',
 'wikimedia_commons',
 'addr:hamlet',
 'payment:cash',
 'wheelchair',
 'historic',
 'fhrs:id',
 'addr:district',
 'addr:subdistrict',
 'contact:email',
 'contact:phone',
 'building:levels',
 'image',
 'contact:facebook',
 'check_date:opening_hours',
 'seasonal',
 'postal_code',
 'old_name',
 'shop',
 'toilets',
 'automatic_door',
 'contact:website',
 'platform_lift',
 'roof:shape',
 'note',
 'opening_hours:url',
 'not:addr:postcode',
 'ref:GB:uprn',
 'description',
 'fax',
 'HE_ref',
 'heritage',
 'heritage:operator',
 'lis

In [37]:
osm_pois_df.loc[osm_pois_df.name.duplicated(keep=False)].sort_values("name")

,element,id,geometry,building,name,tourism,wikidata,addr:city,addr:country,addr:postcode,...,changing_table:location,source:name,building:wikipedia,type,air_conditioning,contact:instagram,contact:twitter,short_name,source:addr:postcode,source:building
180,way,213187266,"POLYGON ((-1.68691 55.40229, -1.68692 55.40232...",yes,NaN,museum,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
181,way,216080122,"POLYGON ((-1.93676 55.61878, -1.93658 55.61881...",yes,NaN,museum,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
186,way,286669519,"POLYGON ((-1.66118 55.43104, -1.66127 55.43103...",yes,NaN,museum,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
187,way,286669520,"POLYGON ((-1.66104 55.43136, -1.66092 55.43139...",yes,NaN,museum,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
188,way,290460483,"POLYGON ((-1.81869 55.40279, -1.81868 55.40264...",yes,NaN,museum,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
189,way,291227659,"POLYGON ((-1.71136 55.29952, -1.7111 55.29955,...",yes,NaN,museum,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
557,way,1207896509,"POLYGON ((-2.97923 54.44899, -2.9792 54.449, -...",yes,NaN,museum,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Load Google Maps POIs

In [52]:
pilot_museums_pids = set( pd.read_parquet("../data/place_ids/28-02-2026_101817_museum.parquet").place_ids )
print(len(pilot_museums_pids))

788


In [58]:
dir = "../data/place_ids"
files = [os.path.join(dir, fn) for fn in os.listdir(dir) if fn.find("pid_details")>=0]
files = pd.Series(files).sort_values(ascending=False).tolist()
print(files)

places_df = pd.read_parquet(files)
places_df['latitude'] = places_df.location.progress_apply(lambda l: l.get("latitude"))
places_df['longitude'] = places_df.location.progress_apply(lambda l: l.get("longitude"))
places_df['name_text'] = places_df.displayName.progress_apply(lambda l: l.get("text") if isinstance(l, dict) else None)
places_df.sort_values('name_text', inplace=True)
places_df.reset_index(names='pid', inplace=True)
places_df.drop_duplicates(subset=['pid', 'latitude', 'longitude'], keep='first', inplace=True)
places_df.reset_index(drop=True, inplace=True)
places_df.head()
places_df.info()

['../data/place_ids/pilot_pid_details_pro.parquet', '../data/place_ids/full_pid_details_pro_mar.parquet', '../data/place_ids/full_pid_details_pro_feb.parquet', '../data/place_ids/full_pid_details_ess_mar.parquet', '../data/place_ids/full_pid_details_ess_feb2.parquet', '../data/place_ids/full_pid_details_ess_feb.parquet']


100%|██████████| 41974/41974 [00:00<00:00, 3003116.80it/s]

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36582 entries, 0 to 36581
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   pid                     36582 non-null  object 
 1   types                   36582 non-null  object 
 2   formattedAddress        36582 non-null  object 
 3   plusCode                35641 non-null  object 
 4   location                36582 non-null  object 
 5   googleMapsUri           8563 non-null   object 
 6   businessStatus          8563 non-null   object 
 7   displayName             8563 non-null   object 
 8   primaryTypeDisplayName  8439 non-null   object 
 9   primaryType             8439 non-null   object 
 10  shortFormattedAddress   36574 non-null  object 
 11  accessibilityOptions    5092 non-null   object 
 12  googleMapsLinks         8563 non-null   object 
 13  postalAddress           36476 non-null  object 
 14  containingPlaces        1559 non-null 

In [62]:
pilot_museum_gmap_pois = places_df.loc[places_df.pid.isin(pilot_museums_pids)].explode('types')
pilot_museum_gmap_pois = pilot_museum_gmap_pois.loc[pilot_museum_gmap_pois.types=='museum'].drop_duplicates(subset='pid')
pilot_museum_gmap_pois.info()

<class 'pandas.core.frame.DataFrame'>
Index: 316 entries, 71 to 8556
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   pid                     316 non-null    object 
 1   types                   316 non-null    object 
 2   formattedAddress        316 non-null    object 
 3   plusCode                314 non-null    object 
 4   location                316 non-null    object 
 5   googleMapsUri           316 non-null    object 
 6   businessStatus          316 non-null    object 
 7   displayName             316 non-null    object 
 8   primaryTypeDisplayName  311 non-null    object 
 9   primaryType             311 non-null    object 
 10  shortFormattedAddress   316 non-null    object 
 11  accessibilityOptions    273 non-null    object 
 12  googleMapsLinks         316 non-null    object 
 13  postalAddress           316 non-null    object 
 14  containingPlaces        75 non-null     objec

In [63]:
pilot_museum_gmap_pois

,pid,types,formattedAddress,plusCode,location,googleMapsUri,businessStatus,displayName,primaryTypeDisplayName,primaryType,shortFormattedAddress,accessibilityOptions,googleMapsLinks,postalAddress,containingPlaces,latitude,longitude,name_text
71,ChIJl_7usNdYeUgRL7ZsKQpmlXE,museum,"Abbey Walk, Abbey Rd, Kirkstall, Leeds LS5 3EH...","{'compoundCode': 'R9CV+V3 Leeds, UK', 'globalC...","{'latitude': 53.822170899999996, 'longitude': ...",https://maps.google.com/?cid=81845600916464410...,OPERATIONAL,"{'languageCode': 'en', 'text': 'Abbey House Mu...","{'languageCode': 'en-US', 'text': 'Museum'}",museum,"Abbey Walk, Abbey Rd, Kirkstall, Leeds","{'wheelchairAccessibleEntrance': True, 'wheelc...",{'directionsUri': 'https://www.google.com/maps...,"{'addressLines': ['Abbey Walk', 'Abbey Road'],...",None,53.822171,-1.607353,Abbey House Museum
78,ChIJ0WdkY-eNfEgRhrEN_--w3gU,museum,"Kirkland, Kendal LA9 5AL, UK","{'compoundCode': '87F4+5C Kendal, UK', 'global...","{'latitude': 54.322979999999994, 'longitude': ...",https://maps.google.com/?cid=42296995984216512...,OPERATIONAL,"{'languageCode': 'en', 'text': 'Abbot Hall'}","{'languageCode': 'en-US', 'text': 'Art Gallery'}",art_gallery,"Kirkland, Kendal","{'wheelchairAccessibleEntrance': True, 'wheelc...",{'directionsUri': 'https://www.google.com/maps...,"{'addressLines': ['Kirkland'], 'languageCode':...",None,54.322980,-2.743988,Abbot Hall
157,ChIJubFsBLMadkgRMioifdPyuGg,museum,"Praed St, London W2 1NY, UK","{'compoundCode': 'GR8G+RQ London, UK', 'global...","{'latitude': 51.5171108, 'longitude': -0.1731131}",https://maps.google.com/?cid=75460481658106783...,OPERATIONAL,"{'languageCode': 'en', 'text': 'Alexander Flem...","{'languageCode': 'en-US', 'text': 'Museum'}",museum,"Clarence Memorial Wing, Praed St, London","{'wheelchairAccessibleEntrance': False, 'wheel...",{'directionsUri': 'https://www.google.com/maps...,"{'addressLines': ['Praed Street'], 'languageCo...","[{'id': 'ChIJP_LG8LIadkgRs3wufNnBwDI', 'name':...",51.517111,-0.173113,Alexander Fleming Museum
176,ChIJKcAt1de1fUgRvfOHlKl19cY,museum,"Allenheads, Hexham NE47 9HN, UK","{'compoundCode': 'RQ2J+X4 Allenheads, UK', 'gl...","{'latitude': 54.8023835, 'longitude': -2.2197423}",https://maps.google.com/?cid=14336494360005702...,OPERATIONAL,"{'languageCode': 'en', 'text': 'Allenheads Her...","{'languageCode': 'en-US', 'text': 'Museum'}",museum,"Allenheads, Hexham","{'wheelchairAccessibleEntrance': True, 'wheelc...",{'directionsUri': 'https://www.google.com/maps...,"{'addressLines': ['Allenheads'], 'languageCode...",None,54.802383,-2.219742,Allenheads Heritage Centre
243,ChIJ2_19mdYEdkgRadLE5rfxLPU,museum,"149 Piccadilly, London W1J 7NT, UK","{'compoundCode': 'GR3X+98 London, UK', 'global...","{'latitude': 51.503473299999996, 'longitude': ...",https://maps.google.com/?cid=17666761210420580...,OPERATIONAL,"{'languageCode': 'en', 'text': 'Apsley House'}","{'languageCode': 'en', 'text': 'Art museum'}",art_museum,"149 Piccadilly, London","{'wheelchairAccessibleEntrance': None, 'wheelc...",{'directionsUri': 'https://www.google.com/maps...,"{'addressLines': ['149 Piccadilly'], 'language...",None,51.503473,-0.151671,Apsley House
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8480,ChIJ28jJcG37fEgRot2HYawGyKo,museum,"Penrith CA10 2FX, UK","{'compoundCode': 'J6WP+F8 Penrith, UK', 'globa...","{'latitude': 54.6462222, 'longitude': -2.7641389}",https://maps.google.com/?cid=12306093319230315...,OPERATIONAL,"{'languageCode': 'en', 'text': 'Yanwath Hall'}","{'languageCode': 'en-US', 'text': 'Museum'}",museum,Penrith,"{'wheelchairAccessibleEntrance': False, 'wheel...",{'directionsUri': 'https://www.google.com/maps...,"{'addressLines': None, 'languageCode': 'en-US'...",None,54.646222,-2.764139,Yanwath Hall
8497,ChIJdxvVAKgxeUgRPpnPDg9rCWQ,museum,"Tower St, York YO1 9RY, UK","{'compoundCode': 'XW4C+3M York, UK', 'globalCo...","{'latitude': 53.9552439, 'longitude': -1.0783677}",https://maps.google.com/?cid=72084103910003571...,OPER

### Match POIs

In [85]:
#i have 2 sets of point of interests with some geometry information. one set consists of only points with lat long information, the other is mix of points and polygons. Both sets have a text name of the POI. These names may not match exactly. Both sets also have a unique ID based on the platform they were sourced from. How can I match these 2 sets to identify the same POIs in the two datasets?

import geopandas as gpd
from shapely.geometry import Point
from rapidfuzz import fuzz, process

def match_poi_datasets(df_a, df_b, distance_threshold=500, score_cutoff=70):
    """
    df_a: GeoDataFrame with points (Lat/Long)
    df_b: GeoDataFrame with mix of points and polygons
    distance_threshold: Max distance in meters to consider a match
    score_cutoff: Minimum fuzzy name match score (0-100)
    """
    
    # 1. Ensure both are in a projected CRS (meters) for accurate distance
    df_a = df_a.to_crs(epsg=3857)
    df_b = df_b.to_crs(epsg=3857)

    # 2. Spatial Join: Find all candidates in B within X meters of A
    # 'inner' join keeps only pairs that are spatially close
    candidates = gpd.sjoin_nearest(
        df_a, 
        df_b, 
        max_distance=distance_threshold, 
        distance_col="dist_meters",
        lsuffix="set_a", 
        rsuffix="set_b"
    )

    matches = []

    # 3. Fuzzy Name Matching on the candidates
    for idx, row in candidates.iterrows():
        name_a = str(row['name_set_a']).lower().strip()
        name_b = str(row['name_set_b']).lower().strip()
        
        # Calculate Token Sort Ratio (handles "Pizza Hut" vs "Hut Pizza")
        name_score = fuzz.token_sort_ratio(name_a, name_b)
        
        if name_score >= score_cutoff:
            matches.append({
                'id_a': row['id_set_a'],
                'id_b': row['id_set_b'],
                'name_a': row['name_set_a'],
                'name_b': row['name_set_b'],
                'distance': row['dist_meters'],
                'name_score': name_score
            })

    return matches

# --- Example Usage ---
# df_points = gpd.read_file("set_a.geojson")
# df_mixed = gpd.read_file("set_b.geojson")
# results = match_poi_datasets(df_points, df_mixed)

In [92]:
results = match_poi_datasets(
    gpd.GeoDataFrame(
        pilot_museum_gmap_pois.rename(
            columns={'name_text':'name', 'pid': 'id'}
        ), geometry=gpd.points_from_xy(pilot_museum_gmap_pois.longitude, pilot_museum_gmap_pois.latitude),
        crs="EPSG:4326"
    ),
    gpd.GeoDataFrame(osm_pois_df, geometry="geometry", crs="EPSG:4326"),
    distance_threshold=500,
    score_cutoff=0
)

pd.DataFrame(results).name_score.describe()

count    137.000000
mean      71.473424
std       27.097986
min       15.384615
25%       44.776119
50%       80.000000
75%      100.000000
max      100.000000
Name: name_score, dtype: float64

In [94]:
pd.DataFrame(results).to_clipboard(index=False)

In [69]:
osm_pois_df.to_clipboard(index=False)

In [70]:
pilot_museum_gmap_pois.to_clipboard(index=False)

In [66]:
len( set(osm_pois_df.name) - set(pilot_museum_gmap_pois.name_text) )

132

### Compare against newly extracted pilot museum searches

In [131]:
missing_pids = """ChIJXcTfP6-lfUgRqECG-3Tf35c
ChIJq6rmlOtZh0gRYtLE2R5mJq4
ChIJ-1kCJLpNh0gR_u3L4uCwQ4w
ChIJl4CfDKSVfUgRn0WFzc4U5Jc
ChIJV18Q9kBbh0gR9SR6ykYBkD0
ChIJzwBi3UXGfUgR2hV_cKpqYho
ChIJZY8PabvZfUgRhNZqdQaZUAI
ChIJ6-Xd5Fhbh0gRD7-rew1Wc3g
ChIJ2-yrR0_GfUgR35X-T30TPTw
ChIJL_aHpUVbh0gR4E7YAlZws6c
ChIJH1TsQEQFfkgRZZc289xi-SA
ChIJRXQbLX7gfUgRYJJvBIhzZ9A
ChIJRRdnmcKNfEgRBSwdBMduj_o
ChIJ_wdeXKO9fEgR48uH3Ew75dg
ChIJRz8VGdvBfEgRv2GhBmiQJJU
ChIJqaCxBdFcfEgRZFVGlaouvF0
ChIJ1_xcbz6WfEgR_etkYa9tnkI
ChIJKTZgx9-_fEgRRDd7s_yO0SE
ChIJpSuXf1GvfEgRfT6FuGHyy_M
ChIJi5MnFpb4fEgRPngCeogdbfU
ChIJjTE-d7qafEgRRjz-7WJ09UU
ChIJR2pe5JykfEgRa0c6HKUX_GY
ChIJQyv-kbsXfEgRaYV4k8cYbVM
ChIJkUzXAEezfkgRX52AxbOVCzA
ChIJw33UqJkffEgRWEq_5V5W8OI
ChIJ89oG0Ua3fkgRXQrggLgw19o
ChIJI9OgXBPpfkgRwBjAZR8XjiY
ChIJdRmfRN5Of0gR9zQ_QIPsnIM
ChIJdWHFoFwqf0gRXW31kSfKgjY
ChIJm3oy69l3fEgRdnmRfeSABnI
ChIJhddgeEusfkgRAZ5w9S41yiY
ChIJtU6EUwAXf0gRc4frO2tMm0I
ChIJ__3s95cffEgRyBn4EPjF8Ig
ChIJZYqp0cFIf0gRgRJSH3Jo8mo
ChIJhU_FUG0Xf0gRDCn-FkhohSE
ChIJNcMDRXXZfkgRzWt3_ctfq9U
ChIJPcn1U4jFfkgRpIdoj-uAk7Q
ChIJ6_kVyYcof0gRiEBP3PXyKIA
ChIJ7c49qT9sfEgRKbtbplFwV18
ChIJH_wjTZRHf0gRltL1z5N9-DA
ChIJ2ceCVmvJfkgRkp7RfHjGaXE
ChIJN1hyRAB3fEgRr1cCb6260Y4
ChIJ9wgWWNM9f0gRiU3INSYDPt0
ChIJuTa_GxNSeUgRI4v2FXmU2ls
ChIJ1YtVmJ67cEgR10Ftyc5yePs
ChIJfcW6xoO8cEgRGMlgCLp27Z4
ChIJo0dMkYy8cEgRRJR-gJ-142Y
ChIJkUjbFyK_cEgR0F8gj3ubtDY
ChIJm_N5-yZDeUgRDATgmdaUJmM
ChIJLT2I8s5deUgR90SpTPQOKKo
ChIJkUE2LuNZeUgR-3lrSzxIFx4
ChIJC-92a6FeeUgRhjTKiBkZbXY
ChIJRXsx0JBIeUgRriPjk55eSg0
ChIJWf5N07IEdkgRA2xSIRaH0r4
ChIJU-XyvqocdkgRFDIVMuDo-SU
ChIJsQVJQ_MPdkgRyLbAaiB1KtY""".split("\n")

new_places_df = pd.read_parquet(
    [os.path.join("../data/place_ids",fn) for fn in os.listdir("../data/place_ids") if fn.startswith("23-04-2026_204939")]
)

set(missing_pids) - set(new_places_df.place_ids)

{'ChIJ1YtVmJ67cEgR10Ftyc5yePs',
 'ChIJ9wgWWNM9f0gRiU3INSYDPt0',
 'ChIJKTZgx9-_fEgRRDd7s_yO0SE',
 'ChIJR2pe5JykfEgRa0c6HKUX_GY',
 'ChIJRRdnmcKNfEgRBSwdBMduj_o',
 'ChIJRXQbLX7gfUgRYJJvBIhzZ9A',
 'ChIJRXsx0JBIeUgRriPjk55eSg0',
 'ChIJdWHFoFwqf0gRXW31kSfKgjY',
 'ChIJfcW6xoO8cEgRGMlgCLp27Z4',
 'ChIJi5MnFpb4fEgRPngCeogdbfU',
 'ChIJjTE-d7qafEgRRjz-7WJ09UU',
 'ChIJkUE2LuNZeUgR-3lrSzxIFx4',
 'ChIJm3oy69l3fEgRdnmRfeSABnI',
 'ChIJqaCxBdFcfEgRZFVGlaouvF0'}